# S&P 500 Options: LSTM

This notebook fits the declared LSTM member of the sequence population snapshotted by
`09_deep_learning`. Chronological windows, validation gaps, checkpoints, and prediction
eligibility are resolved through the shared sequence boundary.

Prerequisite: `09_deep_learning` must create the complete official sequence population.

In [1]:
"""Fit the declared S&P 500 options LSTM request."""

import polars as pl

from case_studies.sp500_options.research_workflow import (
    ALL_LABELS,
    declared_dl_device,
    model_request_catalog,
    open_study,
    published_dl_device,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_subset,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
DEVICE: str = ""

POPULATION_NAME: str = ""

### The device the population was fitted on

A network trained on a GPU and the same network trained on a CPU accumulate their sums in a
different order and reach different weights, so the device is part of what the fitted model is
and sits inside the training identity rather than beside it. The device this population was
fitted on is declared once, in `modeling.dl.device` in `config/setup.yaml`, and read from there
by all four deep-learning notebooks rather than retyped in each. On a machine with no NVIDIA
card the run stops here rather than quietly training something else: set `DEVICE="cpu"` and pass
a `POPULATION_NAME` to fit the same requests there, under a name of their own.

In [3]:
CANONICAL_POPULATION_NAME = "sp500-options-sequence-validation-v1"

published_device = published_dl_device()
device = declared_dl_device(DEVICE)
population_name = POPULATION_NAME or CANONICAL_POPULATION_NAME
if device != published_device and population_name == CANONICAL_POPULATION_NAME:
    raise ValueError(
        f"this run fits on {device!r}, not the published {published_device!r}, so its "
        f"identities are not the ones {CANONICAL_POPULATION_NAME!r} holds; pass "
        f"POPULATION_NAME to give them a population of their own"
    )
print(f"training device: {device} (declared: {published_device})")

training device: cuda (declared: cuda)


## Declared request

In [4]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
requests = model_request_catalog(
    "deep_learning",
    labels=ALL_LABELS,
    config_names=("lstm_h64",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""regression""",52,248,42186,2,2019-01-07 00:00:00,2020-11-10 00:00:00,20,"""canonical""","""7508101644c0"""


## Execute and validate

The shared sequence runner owns chronological window construction, fold fitting, fitted-state
reload, checkpoint publication, restart, and exact eligible-key validation.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_subset(
        study,
        resolved,
        population=population_name,
    )
else:
    if not WORKSPACE or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

Fold-major CV: 2 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...


    train=49,629 seq across 472 symbols
    val=12,326 seq across 473 symbols
    creating datasets...
    datasets ready
    lstm_h64:


      epoch   1/100: train_loss=0.608235


      epoch   2/100: train_loss=0.576638


      epoch   3/100: train_loss=0.539995


...

      epoch  99/100: train_loss=0.130493


      epoch 100/100: train_loss=0.130638, val_loss=0.927717, IC=-0.0223


      best_ep=5, IC=+0.0060 (138.3s, 20 checkpoints)


  lstm_h64: best_epoch=10, IC=+0.0231 (318.5s)



  Best: lstm_h64 @ epoch 10 (IC=+0.0231)
  Saved to ~/ml4t/public-s6-sp500_options/case_studies/sp500_options/run_log/training/7508101644c0/diagnostics


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("LSTM execution returned a partial checkpoint")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",5,"""canonical""",true,"""7508101644c0""","""e6440eaed198"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",10,"""canonical""",true,"""7508101644c0""","""1706327c649d"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",15,"""canonical""",true,"""7508101644c0""","""26046cbbe5e7"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",20,"""canonical""",true,"""7508101644c0""","""399684346173"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",25,"""canonical""",true,"""7508101644c0""","""d6433f19ff36"""
…,…,…,…,…,…,…,…,…
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",80,"""canonical""",true,"""7508101644c0""","""f6fd866de196"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",85,"""canonical""",true,"""7508101644c0""","""72983f4f320f"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",90,"""canonical""",true,"""7508101644c0""","""91074ea85dae"""


The complete LSTM checkpoint population is ready for model analysis and backtesting. This
notebook does not compare it with another family or choose a checkpoint.